Para implementar rigurosamente las 6 recomendaciones estadísticas que discutimos, he desarrollado un script completo en Python. El algoritmo realiza los siguientes pasos secuenciales:

1. **Carga los datos originales** desde la ruta *raw*.
2. **Separa las variables**: Excluye las variables de identificación (`fecha`, `año`, `semana_epi`) y, lo más importante, **excluye la variable objetivo (`casos_dengue`) y sus 12 rezagos** del proceso de PCA para evitar fuga de información (*data leakage*).
3. **Estandariza (Z-score)** las variables meteorológicas remanentes y exporta este dataset escalado.
4. **Aplica PCA** y evalúa de forma iterativa cuántos componentes principales incluir en un modelo ARIMAX. El algoritmo ajusta un modelo ARIMAX utilizando los rezagos del dengue de forma endógena y los componentes principales como variables exógenas ($X$), identificando la cantidad exacta de componentes que minimizan el criterio **AIC** y **BIC**.
5. **Exporta los datasets procesados** (escalado y reducido óptimo) así como un reporte en Excel con las métricas de selección en las rutas que especificaste.

Aquí tienes el código listo para ejecutar:




In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS Y DIRECTORIOS
# ==========================================
ruta_input = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\2_datos\1_raw\2_meteo_epi_rezagos_2026_1.xlsx"
dir_procesados = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\2_datos\2_procesados"
dir_resultados = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\3_resultados"

# Crear los directorios si no existen
os.makedirs(dir_procesados, exist_ok=True)
os.makedirs(dir_resultados, exist_ok=True)

print("Cargando datos...")
df = pd.read_excel(ruta_input)

# Asegurar orden cronológico si existe columna fecha
if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)

# ==========================================
# 2. SEPARACIÓN DE VARIABLES (RECOMENDACIÓN 1)
# ==========================================
# Identificar variables de Dengue (Variable endógena y sus rezagos)
cols_dengue = [col for col in df.columns if 'casos_dengue' in col]

# Identificar variables de control/tiempo que no van al PCA
cols_control = ['fecha', 'año', 'semana_epi']
cols_excluir = cols_dengue + [c for c in cols_control if c in df.columns]

# Variables meteorológicas (las que entrarán al PCA)
cols_meteorologicas = [col for col in df.columns if col not in cols_excluir]

print(f"Total variables meteorológicas a reducir (incluyendo sus rezagos): {len(cols_meteorologicas)}")
print(f"Variables de Dengue protegidas (fuera del PCA): {cols_dengue}")

# ==========================================
# 3. ESTANDARIZACIÓN RIGUROSA (RECOMENDACIÓN 2)
# ==========================================
scaler = StandardScaler()
meteo_escalado = scaler.fit_transform(df[cols_meteorologicas])

# Reconstruir Dataframe Escalado manteniendo las columnas de identificación y dengue
df_escalado = df[cols_excluir].copy()
df_meteo_escalado = pd.DataFrame(meteo_escalado, columns=cols_meteorologicas)
df_escalado = pd.concat([df_escalado, df_meteo_escalado], axis=1)

# Guardar dataset escalado
ruta_escalado = os.path.join(dir_procesados, "datos_meteorologicos_escalados.xlsx")
df_escalado.to_excel(ruta_escalado, index=False)
print(f"Dataset escalado guardado exitosamente en: {ruta_escalado}")

# ==========================================
# 4. TRATAMIENTO DE ESTACIONARIEDAD (RECOMENDACIÓN 4)
# ==========================================
# Nota metodológica: Para asegurar la convergencia de los modelos autorregresivos,
# trabajaremos con las primeras diferencias para estabilizar la media si fuera necesario.
# Ajustaremos un modelo ARIMAX(1,1,1) base para evaluar el aporte de los componentes (X).
order_arimax = (1, 1, 1) 

y = df['casos_dengue'] # Variable endógena (en niveles, el parámetro 'd=1' del ARIMAX la diferenciará)

# ==========================================
# 5. ALGORITMO DE SELECCIÓN POR AIC / BIC (RECOMENDACIÓN 3, 5 Y 6)
# ==========================================
# Aplicamos el PCA completo primero para evaluar todas las combinaciones
pca_completo = PCA()
meteo_pca = pca_completo.fit_transform(meteo_escalado)

# Lista para almacenar los resultados de cada configuración
resultados_lista = []

# Probar desde 1 componente hasta un máximo razonable (ej. 15 componentes o el total disponible)
max_componentes = min(15, len(cols_meteorologicas))

print("\nIniciando optimización de componentes mediante ARIMAX...")
for n_comp in range(1, max_componentes + 1):
    # Seleccionar los primeros 'n_comp' componentes
    X_pca = meteo_pca[:, :n_comp]
    
    # Columnas de nombres para el dataframe de componentes
    cols_pc = [f"PC_{i+1}" for i in range(n_comp)]
    df_exog = pd.DataFrame(X_pca, columns=cols_pc)
    
    try:
        # Ajustar modelo ARIMAX empleando los componentes como variables exógenas
        # statsmodels aplica la estructura ARMA sobre los residuos tras restar el efecto de X_pca
        modelo = SARIMAX(y, exog=df_exog, order=order_arimax, 
                         enforce_stationarity=False, enforce_invertibility=False)
        resultado_modelo = modelo.fit(disp=False)
        
        # Varianza explicada acumulada de los componentes actuales
        var_acumulada = np.sum(pca_completo.explained_variance_ratio_[:n_comp]) * 100
        
        resultados_lista.append({
            'Num_Componentes': n_comp,
            'AIC': resultado_modelo.aic,
            'BIC': resultado_modelo.bic,
            'Log-Likelihood': resultado_modelo.llf,
            'Varianza_Explicada_Acum_%': round(var_acumulada, 2)
        })
    except Exception as e:
        # En caso de que alguna combinación matemática de componentes no converja
        continue

# Crear Dataframe con los resultados del análisis estadístico
df_resultados_pca = pd.DataFrame(resultados_lista)

# Determinar las mejores combinaciones según criterios de información
mejor_aic = df_resultados_pca.loc[df_resultados_pca['AIC'].idxmin()]
mejor_bic = df_resultados_pca.loc[df_resultados_pca['BIC'].idxmin()]

print("\n--- RESULTADOS DEL ANÁLISIS ---")
print(f"Mejor configuración según AIC: {int(mejor_aic['Num_Componentes'])} componentes (AIC: {mejor_aic['AIC']:.2f}, Var. Explicada: {mejor_aic['Varianza_Explicada_Acum_%']}%)")
print(f"Mejor configuración según BIC (Más parsimonioso): {int(mejor_bic['Num_Componentes'])} componentes (BIC: {mejor_bic['BIC']:.2f}, Var. Explicada: {mejor_bic['Varianza_Explicada_Acum_%']}%)")

# ==========================================
# 6. EXPORTACIÓN DE DATASETS Y REPORTES FINALES
# ==========================================
# Elegimos el criterio AIC por defecto para guardar el dataset reducido (puedes cambiarlo a mejor_bic si buscas parsimonia extrema)
n_optimo = int(mejor_aic['Num_Componentes'])

X_optimo = meteo_pca[:, :n_optimo]
cols_pc_optimas = [f"PC_{i+1}" for i in range(n_optimo)]
df_pca_optimo = pd.DataFrame(X_optimo, columns=cols_pc_optimas)

# Dataset final reducido: ID + Dengue y Rezagos + Componentes Principales Óptimos
df_reducido_final = pd.concat([df[cols_excluir], df_pca_optimo], axis=1)

# Guardar dataset reducido
ruta_reducido = os.path.join(dir_procesados, "datos_meteorologicos_reducidos_pca.xlsx")
df_reducido_final.to_excel(ruta_reducido, index=False)
print(f"\nDataset reducido final (con {n_optimo} componentes) guardado en: {ruta_reducido}")

# Guardar matriz de cargas de los componentes (Loadings) para el análisis epidemiológico (Recomendación 5)
cargas_pca = pd.DataFrame(
    pca_completo.components_[:n_optimo].T, 
    columns=cols_pc_optimas, 
    index=cols_meteorologicas
)

ruta_resultados_excel = os.path.join(dir_resultados, "reporte_seleccion_pca_arimax.xlsx")

# Escribir múltiples hojas en el excel de resultados
with pd.ExcelWriter(ruta_resultados_excel) as writer:
    df_resultados_pca.to_excel(writer, sheet_name='Criterios_AIC_BIC', index=False)
    cargas_pca.to_excel(writer, sheet_name='Cargas_Variables_Loadings')

print(f"Reporte estadístico de resultados guardado exitosamente en: {ruta_resultados_excel}")
print("¡Proceso completado con éxito!")


Cargando datos...
Total variables meteorológicas a reducir (incluyendo sus rezagos): 156
Variables de Dengue protegidas (fuera del PCA): ['casos_dengue', 'casos_dengue_lag_1', 'casos_dengue_lag_2', 'casos_dengue_lag_3', 'casos_dengue_lag_4', 'casos_dengue_lag_5', 'casos_dengue_lag_6', 'casos_dengue_lag_7', 'casos_dengue_lag_8', 'casos_dengue_lag_9', 'casos_dengue_lag_10', 'casos_dengue_lag_11', 'casos_dengue_lag_12']
Dataset escalado guardado exitosamente en: C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\2_datos\2_procesados\datos_meteorologicos_escalados.xlsx

Iniciando optimización de componentes mediante ARIMAX...

--- RESULTADOS DEL ANÁLISIS ---
Mejor configuración según AIC: 1 componentes (AIC: 1921.65, Var. Explicada: 37.86%)
Mejor configuración según BIC (Más parsimonioso): 1 componentes (BIC: 1936.00, Var. Explicada: 37.86%)

Dataset reducido final (con 1 componentes) guardado en: C:\Users\marco\Documentos\investigacion\arima\06_


# Características destacadas del script:

1. **Evita bucles de error:** Utiliza bloques `try-except` dentro de la optimización del ARIMAX por si alguna matriz de componentes provoca problemas de multicolinealidad o singularidad extrema durante el ajuste.
2. **Interpretabilidad Epidemiológica:** Guarda automáticamente una pestaña llamada `Cargas_Variables_Loadings` en el Excel de resultados. Allí podrás inspeccionar qué variables climáticas reales y qué rezagos específicos tienen más peso en los componentes elegidos.
3. **Manejo estricto de rutas:** El uso de prefijos `r` en los strings de las rutas evita errores comunes de escape en Windows provocados por las barras invertidas (`\`).

# Entrenamiento final 



Para llevar a cabo el entrenamiento final de tu modelo ARIMAX con el dataset reducido por PCA, es fundamental seguir un proceso estadístico estricto para series temporales:

1. **Definición de Variables:** La variable dependiente ($y$) será `casos_dengue` y la exógena ($X$) será `PC_1`.
2. **Validación de Estacionariedad:** Evaluaremos si las series necesitan diferenciación utilizando la prueba de Dickey-Fuller Aumentada (ADF).
3. **Identificación de Órdenes ($p, d, q$):** Utilizaremos las funciones de autocorrelación (ACF) y autocorrelación parcial (PACF) para guiar la selección de los rezagos autorregresivos ($p$) y de medias móviles ($q$).
4. **Entrenamiento y Diagnóstico de Residuos:** Ajustaremos el modelo final y analizaremos sus residuos mediante la prueba de Ljung-Box para asegurar que se comporten como ruido blanco (lo que garantiza que el modelo extrajo toda la información disponible).

Aquí tienes el script completo en Python estructurado profesionalmente:

```python


In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS Y CARGA DE DATOS
# ==========================================
dir_procesados = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\2_datos\2_procesados"
dir_resultados = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\3_resultados"

ruta_dataset = os.path.join(dir_procesados, "datos_meteorologicos_reducidos_pca.xlsx")

print("Cargando el dataset reducido...")
df = pd.read_excel(ruta_dataset)

# Asegurar el orden cronológico
if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)

# Definir variables clave
y = df['casos_dengue']
X = df[['PC_1']]  # Debe ser un DataFrame o matriz bidimensional para statsmodels

# ==========================================
# 2. PRUEBA DE ESTACIONARIEDAD (ADF TEST)
# ==========================================
print("\n--- Evaluando Estacionariedad (Prueba Dickey-Fuller Aumentada) ---")
def verificar_estacionariedad(serie, nombre):
    resultado = adfuller(serie.dropna())
    print(f"Variable: {nombre}")
    print(f"  Estadístico ADF: {resultado[0]:.4f}")
    print(f"  p-valor: {resultado[1]:.4f}")
    if resultado[1] < 0.05:
        print("  Resultado: Estacionaria (Rechaza H0)")
        return 0
    else:
        print("  Resultado: NO Estacionaria (No rechaza H0). Requiere diferenciación.")
        return 1

d_y = verificar_estacionariedad(y, "casos_dengue")
d_x = verificar_estacionariedad(X['PC_1'], "PC_1")

# El orden de integración 'd' sugerido para el modelo ARIMAX
d_final = max(d_y, d_x) 
print(f"\nOrden de diferenciación integrado (d) recomendado para el ARIMAX: {d_final}")

# ==========================================
# 3. ANÁLISIS DE CORRELACIÓN (ACF Y PACF)
# ==========================================
# Graficamos la serie (diferenciada si corresponde) para estimar p y q
y_analisis = y.diff(d_final).dropna() if d_final > 0 else y

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_analisis, ax=axes[0], lags=24, title=f"ACF - Dengue (d={d_final})")
plot_pacf(y_analisis, ax=axes[1], lags=24, title=f"PACF - Dengue (d={d_final})")
plt.tight_layout()

ruta_grafico_corr = os.path.join(dir_resultados, "acf_pacf_diagnose.png")
plt.savefig(ruta_grafico_corr)
print(f"Gráficos ACF/PACF guardados en: {ruta_grafico_corr}")
plt.close()

# ==========================================
# 4. AJUSTE DEL MODELO ARIMAX FINAL
# ==========================================
# Nota: Puedes cambiar el orden (p, d, q) basándote en los gráficos ACF/PACF visualizados.
# Usaremos como base un ARIMAX(1, d, 1) que suele ser un estándar sólido de inicio.
p = 1
d = d_final
q = 1

print(f"\nAjustando modelo ARIMAX({p}, {d}, {q}) con variable exógena PC_1...")
modelo_final = SARIMAX(y, exog=X, order=(p, d, q), 
                       enforce_stationarity=False, 
                       enforce_invertibility=False)

res_final = modelo_final.fit(disp=False)

# Mostrar el resumen estadístico formal en consola
print("\n========================================================")
print("             RESUMEN DEL MODELO ARIMAX")
print("========================================================")
print(res_final.summary())

# ==========================================
# 5. DIAGNÓSTICO DE RESIDUOS (RUIDO BLANCO)
# ==========================================
print("\n--- Diagnóstico de Residuos ---")
# Gráficos de diagnóstico del modelo
res_final.plot_diagnostics(figsize=(12, 8))
ruta_grafico_diag = os.path.join(dir_resultados, "diagnostico_residuos_arimax.png")
plt.savefig(ruta_grafico_diag)
print(f"Gráfico de diagnóstico de residuos guardado en: {ruta_grafico_diag}")
plt.close()

# ==========================================
# 6. EXPORTACIÓN DE RESULTADOS FINALES
# ==========================================
# 1. Guardar los coeficientes y métricas del modelo en un archivo de texto
ruta_reporte_txt = os.path.join(dir_resultados, "resumen_modelo_arimax.txt")
with open(ruta_reporte_txt, "w") as f:
    f.write(res_final.summary().as_text())

# 2. Guardar valores ajustados (Predicciones dentro de la muestra) frente a reales
df_predicciones = pd.DataFrame({
    'Fecha': df['fecha'] if 'fecha' in df.columns else df.index,
    'Casos_Reales': y,
    'Casos_Ajustados_Modelo': res_final.fittedvalues
})

ruta_predicciones = os.path.join(dir_resultados, "valores_ajustados_arimax.xlsx")
df_predicciones.to_excel(ruta_predicciones, index=False)

print(f"\nReporte de texto guardado en: {ruta_reporte_txt}")
print(f"Valores ajustados (reales vs estimados) guardados en: {ruta_predicciones}")
print("¡Entrenamiento del modelo final ARIMAX finalizado con éxito!")


Cargando el dataset reducido...

--- Evaluando Estacionariedad (Prueba Dickey-Fuller Aumentada) ---
Variable: casos_dengue
  Estadístico ADF: -2.5046
  p-valor: 0.1144
  Resultado: NO Estacionaria (No rechaza H0). Requiere diferenciación.
Variable: PC_1
  Estadístico ADF: -3.2904
  p-valor: 0.0153
  Resultado: Estacionaria (Rechaza H0)

Orden de diferenciación integrado (d) recomendado para el ARIMAX: 1
Gráficos ACF/PACF guardados en: C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\3_resultados\acf_pacf_diagnose.png

Ajustando modelo ARIMAX(1, 1, 1) con variable exógena PC_1...

             RESUMEN DEL MODELO ARIMAX
                               SARIMAX Results                                
Dep. Variable:           casos_dengue   No. Observations:                  270
Model:               SARIMAX(1, 1, 1)   Log Likelihood                -956.827
Date:                Wed, 24 Jun 2026   AIC                           1921.654
Time:       

### ¿Qué evaluar en los resultados de este script?

Cuando el script finalice, se generará el resumen formal en tu consola y un archivo de texto. Debes fijarte primordialmente en tres puntos críticos:

1. **El coeficiente de `PC_1`:** En la tabla de coeficientes, busca la fila `PC_1`. Mira su valor `P>|z|` (p-valor). Si es **menor a 0.05**, significa que tu componente climático reducido tiene un impacto estadísticamente significativo para explicar las variaciones en los casos de dengue.
2. **Prueba de Ljung-Box (Prob(Q)):** En la parte inferior del resumen verás la estadística `Prob(Q)`. Si este valor es **mayor a 0.05**, significa que los residuos no están correlacionados entre sí; pasaron la prueba de ruido blanco, validando que el modelo es estadísticamente óptimo.
3. **El gráfico `diagnostico_residuos_arimax.png`:** La esquina superior derecha (gráfico QQ-plot) debe mostrar los puntos alineados lo más cerca posible a la línea recta diagonal, lo cual confirmará que los errores del modelo siguen una distribución normal.

# Medida de desempeño mae y graficación de desempeño

Para realizar correctamente la división de datos en entrenamiento y validación (testeo) con datos de series temporales, **no se debe utilizar un muestreo aleatorio**. Dado que las observaciones son dependientes del tiempo, la división debe ser **cronológica**: los primeros datos se usan para entrenar y los últimos datos (los más recientes) se reservan para el testeo.

Además, como solicitas restringir el análisis al intervalo desde el año **2022 hasta el final de los datos**, el algoritmo primero filtrará el dataset por este período histórico y luego calculará los puntos de corte cronológicos para los porcentajes solicitados (80-20, 90-10, 95-5, y 97-3).

A continuación, tienes el script de Python que realiza el filtrado, ejecuta el modelo ARIMAX utilizando el componente `PC_1` en cada una de las particiones temporales, calcula el Error Absoluto Medio (MAE) tanto en el ajuste de entrenamiento como en la predicción de testeo, y genera los gráficos comparativos (líneas y barras) solicitados.

```python


In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS Y CARGA DE DATOS
# ==========================================
dir_procesados = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\2_datos\2_procesados"
dir_resultados = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\3_resultados"

ruta_dataset = os.path.join(dir_procesados, "datos_meteorologicos_reducidos_pca.xlsx")

print("Cargando el dataset reducido por PCA...")
df = pd.read_excel(ruta_dataset)

# Asegurar formato de fecha y ordenamiento cronológico
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

# ==========================================
# 2. FILTRADO TEMPORAL (DESDE 2022 EN ADELANTE)
# ==========================================
print("Filtrando datos desde el año 2022...")
df_filtrado = df[df['fecha'].dt.year >= 2022].reset_index(drop=True)
total_registros = len(df_filtrado)
print(f"Total de semanas/registros evaluados desde 2022: {total_registros}")

# Definición de variables del modelo
y = df_filtrado['casos_dengue']
X = df_filtrado[['PC_1']]

# ==========================================
# 3. EVALUACIÓN DE PARTICIONES (80-20, 90-10, 95-5, 97-3)
# ==========================================
particiones = {
    "80-20": 0.20,
    "90-10": 0.10,
    "95-5":  0.05,
    "97-3":  0.03
}

# Diccionario para almacenar los resultados de las métricas
resultados_mae = []

# Configuración base del ARIMAX determinada previamente (ej. p=1, d=1, q=1)
order_arimax = (1, 1, 1)

print("\nEvaluando particiones cronológicas...")
for nombre_particion, porc_test in particiones.items():
    # Calcular el índice de división (Split) cronológico
    tamano_test = int(np.floor(total_registros * porc_test))
    indice_corte = total_registros - tamano_test
    
    # División de la variable dependiente (Dengue)
    y_train, y_test = y.iloc[:indice_corte], y.iloc[indice_corte:]
    
    # División de la variable exógena (Clima - PC1)
    X_train, X_test = X.iloc[:indice_corte], X.iloc[indice_corte:]
    
    try:
        # Entrenar el modelo con el set de datos histórico (Train)
        modelo = SARIMAX(y_train, exog=X_train, order=order_arimax,
                         enforce_stationarity=False, enforce_invertibility=False)
        resultado_fit = modelo.fit(disp=False)
        
        # 1. MAE de Entrenamiento (valores ajustados dentro de la muestra)
        pred_train = resultado_fit.fittedvalues
        mae_train = mean_absolute_error(y_train, pred_train)
        
        # 2. MAE de Testeo (Pronóstico fuera de la muestra)
        # Especificamos los pasos a predecir y la variable exógena del futuro
        pred_test = resultado_fit.forecast(steps=tamano_test, exog=X_test)
        mae_test = mean_absolute_error(y_test, pred_test)
        
        resultados_mae.append({
            "Partición": nombre_particion,
            "N_Entrenamiento": len(y_train),
            "N_Testeo": len(y_test),
            "MAE_Entrenamiento": round(mae_train, 4),
            "MAE_Testeo": round(mae_test, 4)
        })
        print(f"-> Partición {nombre_particion} evaluada exitosamente.")
        
    except Exception as e:
        print(f"-> Error al evaluar la partición {nombre_particion}: {str(e)}")

# Convertir resultados a DataFrame
df_mae = pd.DataFrame(resultados_mae)
print("\n--- TABLA COMPARATIVA DE ERRORES (MAE) ---")
print(df_mae.to_string(index=False))

# Guardar los resultados numéricos en Excel
ruta_salida_excel = os.path.join(dir_resultados, "metricas_particiones_mae.xlsx")
df_mae.to_excel(ruta_salida_excel, index=False)

# ==========================================
# 4. GENERACIÓN DE GRÁFICOS
# ==========================================
x_labels = df_mae["Partición"]
x_indices = np.arange(len(x_labels))
width = 0.35

# ---- GRÁFICO 1: DIAGRAMA DE BARRAS COMPARATIVO ----
plt.figure(figsize=(10, 6))
plt.bar(x_indices - width/2, df_mae["MAE_Entrenamiento"], width, label='MAE Entrenamiento', color='#3498db')
plt.bar(x_indices + width/2, df_mae["MAE_Testeo"], width, label='MAE Testeo', color='#e74c3c')

plt.xlabel('Estrategia de División (Train-Test)', fontsize=12)
plt.ylabel('Error Absoluto Medio (MAE)', fontsize=12)
plt.title('Comparativo de MAE por Partición Temporal (Desde 2022)\n[Diagrama de Barras]', fontsize=14, fontweight='bold')
plt.xticks(x_indices, x_labels)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)

ruta_barras = os.path.join(dir_resultados, "grafico_barras_mae.png")
plt.savefig(ruta_barras, dpi=300)
print(f"\nGráfico de barras guardado en: {ruta_barras}")
plt.close()

# ---- GRÁFICO 2: GRÁFICO DE LÍNEAS ----
plt.figure(figsize=(10, 6))
plt.plot(x_labels, df_mae["MAE_Entrenamiento"], marker='o', linewidth=2, markersize=8, label='MAE Entrenamiento', color='#2980b9')
plt.plot(x_labels, df_mae["MAE_Testeo"], marker='s', linewidth=2, markersize=8, label='MAE Testeo', color='#c0392b')

plt.xlabel('Estrategia de División (Train-Test)', fontsize=12)
plt.ylabel('Error Absoluto Medio (MAE)', fontsize=12)
plt.title('Tendencia del MAE según Proporción de Datos (Desde 2022)\n[Gráfico de Líneas]', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

ruta_lineas = os.path.join(dir_resultados, "grafico_lineas_mae.png")
plt.savefig(ruta_lineas, dpi=300)
print(f"Gráfico de líneas guardado en: {ruta_lineas}")
plt.close()

print("\n¡Análisis comparativo de particiones finalizado!")


Cargando el dataset reducido por PCA...
Filtrando datos desde el año 2022...
Total de semanas/registros evaluados desde 2022: 230

Evaluando particiones cronológicas...
-> Partición 80-20 evaluada exitosamente.
-> Partición 90-10 evaluada exitosamente.
-> Partición 95-5 evaluada exitosamente.
-> Partición 97-3 evaluada exitosamente.

--- TABLA COMPARATIVA DE ERRORES (MAE) ---
Partición  N_Entrenamiento  N_Testeo  MAE_Entrenamiento  MAE_Testeo
    80-20              184        46             6.5190     16.9050
    90-10              207        23             6.4697     23.7157
     95-5              219        11             6.7342      7.4744
     97-3              224         6             6.6481     12.3226

Gráfico de barras guardado en: C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\1_pca\3_resultados\grafico_barras_mae.png
Gráfico de líneas guardado en: C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensiona


```

### ¿Cómo interpretar las gráficas y resultados que generará este script?

Cuando revises el archivo `metricas_particiones_mae.xlsx` y las imágenes guardadas en la carpeta de resultados, presta atención a la relación entre los dos errores:

1. **Efecto del tamaño del testeo:** Las particiones como la `97-3` evalúan al modelo en un horizonte de tiempo muy corto (apenas las últimas semanas del dataset). Esto suele resultar en un MAE de testeo bajo si la tendencia no cambia drásticamente, pero representa una validación menos exigente.
2. **Evaluación de sobreajuste:** El MAE de entrenamiento medirá qué tan bien se memorizó el pasado el algoritmo. Si el MAE de testeo en la partición `80-20` (que pone a prueba al modelo durante un período largo) se dispara a valores extremadamente altos en comparación con el de entrenamiento, te indicará que el modelo ARIMAX está perdiendo capacidad de generalización ante cambios estacionales climáticos de largo plazo.